# POLSCI 3

## Going Further: Data Wrangling with `dplyr`

**This notebook is optional and not tied to any particular week.** Do it whenever you have time: it teaches new R tools you'll find useful for the rest of the course and beyond.

### What you'll learn

| Function | What it does |
|---|---|
| `%>%` (the pipe) | Chains steps together so code reads left to right |
| `filter()` | Keeps rows that meet a condition |
| `arrange()` | Sorts rows |
| `select()` | Keeps (or drops) columns |
| `mutate()` | Creates new columns |
| `group_by()` + `summarize()` | Computes summaries for each group |

### How it works

Each section has (1) an explanation, (2) **examples on a tiny fake dataset** so you can see every row and column, and (3) **your turn** on the WNBA data. After each exercise, run the check cell (`. = ottr::check(...)`) to see if you're right. Each exercise tells you what to produce and what to name it. You figure out the steps. This is not graded.

In [ ]:
library(tidyverse)   # loads dplyr (this notebook), ggplot2 (plots), and more

wnba <- read_csv('ps3_wnba_teamgames.csv')
wnba

`library(tidyverse)` loads a bundle of packages, including **`dplyr`**, which has all the functions in this notebook. `read_csv()` is the tidyverse version of `read.csv()`: it gives you a **tibble**, which is just a data frame that prints more nicely.

The WNBA data has one row per team per game (2022 to 2025, regular season and playoffs). The columns are `season`, `game_id`, `game_date`, `game_type`, `team`, `home_away`, `points`, `opp_points`, `win`, `reg_win_pct`, `assists`, `rebounds`, `turnovers`, and `threes_made`.

### A tiny fake dataset

For the examples we'll use a small fake dataset of 8 players, so you can see every row. (Note the **`NA`**: Fay's minutes are missing.) Run this cell and look at it carefully.

In [ ]:
fake <- tibble(
  player   = c("Ava", "Bea", "Cam", "Dee", "Eli", "Fay", "Gus", "Hal"),
  team     = c("Owls", "Owls", "Owls", "Foxes", "Foxes", "Foxes", "Bears", "Bears"),
  position = c("G", "F", "C", "G", "F", "C", "G", "F"),
  points   = c(18, 12, 6, 22, 9, 14, 5, 11),
  minutes  = c(32, 28, 20, 35, 25, NA, 12, 22),
  starter  = c(TRUE, TRUE, FALSE, TRUE, FALSE, TRUE, FALSE, TRUE)
)
fake

---
## 1. The pipe: `%>%`

Real data work usually takes several steps. Without the pipe you either nest functions inside each other (which you read from the inside out) or save lots of intermediate objects. The **pipe** `%>%` takes whatever is on its left and passes it as the **first input** of the function on its right:

- `x %>% f()` is the same as `f(x)`
- `x %>% f(y)` is the same as `f(x, y)`

Read `%>%` out loud as **"and then."**

In [ ]:
# Without a pipe: read from the inside out
round(mean(fake$points), 1)

# With pipes: read left to right (take the points, THEN take the mean, THEN round)
fake$points %>% mean() %>% round(1)

In [ ]:
# These two lines are the same thing:
head(fake, 3)
fake %>% head(3)

When a chain gets long, put **each step on its own line**, ending each line (except the last) with `%>%`. We'll do that from here on.

(R also has a built-in pipe, `|>`, that works almost the same way. We'll use `%>%` because that's what the course materials use.)

### Your turn

**Q1.** Using pipes (no nested parentheses), take the `points` column of `wnba`, compute its mean, and round to 2 decimal places. Save the result as `q1.avg`.

In [ ]:
# points -> mean -> round
q1.avg <- wnba$points %>% mean() %>% round(2)
q1.avg

In [ ]:
. = ottr::check("tests/q1.R")

In [ ]:
---
## 2. `%in%`: see if a value (or several values) appear in a vector

In [ ]:
c("hello", "I", "am", "Maya") %in% c("I")

In [ ]:
---
The TRUE and FALSE indicates if the entry at that point in the vector directly matches the "I".

In [ ]:
---
You can also check for multiple values in a vector:

In [ ]:
c("cat", "panda", "zebra", "dolphin", "tuna", "lion", "sloth", "zebra", "toucan") %in% c("zebra", "cat")

---
## 2. `filter()`: keep the rows you want

`filter(data, condition)` keeps only the rows where the condition is `TRUE`. With a pipe: `data %>% filter(condition)`.

Conditions are built from comparisons: `==` (equal), `!=` (not equal), `>`, `>=`, `<`, `<=`, and combined with `&` (and), `|` (or), and `!` (not). Remember: **`==` tests equality, `=` assigns.** Two handy extras: `x %in% c(...)` (is `x` one of these values?) and `is.na(x)` (is `x` missing?).

Run each example and compare the output to the `fake` table above.

In [ ]:
fake %>% filter(team == "Owls")                        # one condition

fake %>% filter(points >= 10 & starter)                # AND: both must be true

fake %>% filter(position == "G" | position == "C")     # OR: either can be true

fake %>% filter(position %in% c("G", "C"))             # same as the line above, shorter

fake %>% filter(is.na(minutes))                        # which rows have missing minutes?

fake %>% filter(!is.na(minutes))                       # ! flips it: rows that are NOT missing

Notice that `filter()` doesn't change `fake`. It returns a new, smaller table. To keep the result you need to **save it**: `small <- fake %>% filter(...)`.

### Your turn

**Q2.** Create `playoff.2025`, containing only the **playoff** games from the **2025** season.

In [ ]:
playoff.2025 <- wnba %>%
  filter(game_type == "playoff" & season == 2025)

In [ ]:
---
OR

In [ ]:
playoff.2025 <- wnba %>%
  filter(game_type == "playoff") %>%
  filter(season == 2025)

In [ ]:
. = ottr::check("tests/q2.R")

**Q3.** Create `big.nights`, containing team-games where the team scored **at least 100 points OR made at least 15 three-pointers** (`threes_made`).

In [ ]:
big.nights <- wnba %>%
  filter(points >= 100 | threes_made >= 15)

In [ ]:
. = ottr::check("tests/q3.R")

**Q4.** Create `trio.2024`, containing the 2024 games (regular season and playoffs) played by the **Aces, Liberty, or Lynx**.

In [ ]:
trio.2024 <- wnba %>%
  filter(season == 2024 & team %in% c("Aces", "Liberty", "Lynx"))

In [ ]:
. = ottr::check("tests/q4.R")

---
## 3. `arrange()`: sort the rows

`arrange(data, column)` sorts rows from smallest to largest. Wrap the column in `desc()` to sort from largest to smallest. If you give several columns, later ones break ties in earlier ones. Missing values (`NA`) always go to the end.

`head(n)` (which you've seen) keeps the first `n` rows, so `arrange()` followed by `head()` finds the top or bottom of a dataset.

In [ ]:
fake %>% arrange(points)                       # smallest to largest

fake %>% arrange(desc(points))                 # largest to smallest

fake %>% arrange(team, desc(points))           # by team (A to Z), then highest points within each team

fake %>% arrange(desc(points)) %>% head(3)     # the top 3 scorers

### Your turn

**Q5.** Create `top.scorers`, the **10 team-games with the most points** in the data, ordered from most to fewest.

In [ ]:
top.scorers <- wnba %>%
  arrange(desc(points)) %>%
  head(10)

In [ ]:
. = ottr::check("tests/q5.R")

**Q6.** Which team has the single lowest-scoring game in the data? Save the team's name as `q6.team`.

In [ ]:
lowest <- wnba %>%
  arrange(points) %>%
  head(1)

q6.team <- lowest$team
q6.team

In [ ]:
. = ottr::check("tests/q6.R")

---
## 4. `select()`: keep (or drop) columns

While `filter()` picks **rows**, `select()` picks **columns**. You can name the columns you want (they come out in the order you list them), select a range with `:`, drop columns with `-`, and use helpers like `starts_with()`, `ends_with()`, and `contains()`. `everything()` means all the other columns.

In [ ]:
fake %>% select(player, points)                # just these columns

fake %>% select(points, player)                # the order you list is the order you get

fake %>% select(player:points)                 # a range: everything from player through points

fake %>% select(-minutes, -starter)            # everything EXCEPT these

fake %>% select(team, everything())            # bring team to the front, keep the rest

fake %>% select(starts_with('p'))              # columns whose names start with the letter p

### Your turn

**Q7.** Create `compact`, containing **only** the columns `season`, `team`, `home_away`, and `points` (in that order).

In [ ]:
compact <- wnba %>%
  select(season, team, home_away, points)

In [ ]:
. = ottr::check("tests/q7.R")

**Q8.** Create `no.stats`, which is `wnba` with the four box-score columns `assists`, `rebounds`, `turnovers`, and `threes_made` **removed** (keep everything else).

In [ ]:
no.stats <- wnba %>%
  select(-assists, -rebounds, -turnovers, -threes_made)

In [ ]:
. = ottr::check("tests/q8.R")

---
## 5. `mutate()`: create new columns

`mutate(data, new_column = some calculation)` adds a new column, calculated from existing ones, and keeps all the old columns. You can make several new columns in one `mutate()`. Remember to **save** the result if you want to keep the new column.

In [ ]:
fake %>% mutate(pts_per_min = points / minutes)       # arithmetic on columns (NA stays NA)

fake %>% mutate(ten_plus = points >= 10)              # a TRUE/FALSE column

fake %>% mutate(role = ifelse(starter, "starter", "bench"))   # ifelse(test, if TRUE, if FALSE)

# several new columns at once; later ones can use earlier ones
fake %>% mutate(double_points = points * 2,
                double_plus_one = double_points + 1)

### Your turn

**Q9.** Create `wnba2`, which is `wnba` plus a new column `margin` equal to the team's points minus the opponent's points (so a positive margin means the team won).

In [ ]:
wnba2 <- wnba %>%
  mutate(margin = points - opp_points)

In [ ]:
. = ottr::check("tests/q9.R")

**Q10.** Create `wnba3`, which is `wnba2` plus a column `blowout` that is `TRUE` when the game was decided by **20 or more points** (in either direction) and `FALSE` otherwise. (Hint for the "either direction" part: `abs()` takes an absolute value.)

In [ ]:
wnba3 <- wnba2 %>%
  mutate(blowout = abs(margin) >= 20)

In [ ]:
. = ottr::check("tests/q10.R")

---
## 6. `group_by()` and `summarize()`: summaries for each group

`summarize()` collapses many rows into **one row** of summary numbers, like a mean or a count. On its own it summarizes the whole dataset. Add **`group_by()`** first, and you get **one row per group** instead. Inside `summarize()`, `n()` counts the rows in each group.

If a column has `NA`s, functions like `mean()` return `NA` unless you add `na.rm = TRUE`.

In [ ]:
fake %>% summarize(avg_points = mean(points))                    # one row for the whole dataset

fake %>%
  group_by(team) %>%
  summarize(avg_points = mean(points), n_players = n())            # one row per team

fake %>%
  group_by(team, starter) %>%
  summarize(avg_points = mean(points))                             # one row per team-and-starter combination

fake %>%
  group_by(team) %>%
  summarize(avg_minutes = mean(minutes, na.rm = TRUE))             # ignore the missing minutes

When you group by **two** variables, R prints a harmless message about "grouped output." You can ignore it, or add `.groups = "drop"` inside `summarize()` to silence it.

### Your turn

**Q11.** Create `by.type.home`: the **average points** for each combination of `game_type` and `home_away` (4 rows). Name the new column `avg_points`.

In [ ]:
by.type.home <- wnba %>%
  group_by(game_type, home_away) %>%
  summarize(avg_points = mean(points), .groups = "drop")
by.type.home

In [ ]:
. = ottr::check("tests/q11.R")

**Q12.** Create `team.summary`, with one row per team (across all games in the data) and three columns: `games` (number of team-games), `win_pct` (the share of games won; `win` is 1 for a win and 0 for a loss), and `avg_points`.

In [ ]:
team.summary <- wnba %>%
  group_by(team) %>%
  summarize(games = n(),
            win_pct = mean(win),
            avg_points = mean(points))
team.summary

In [ ]:
. = ottr::check("tests/q12.R")

---
## 7. Putting it all together

The real power of dplyr is chaining these functions with pipes. Here is a chain on the fake data. Run it, then read it top to bottom as a recipe: *start with `fake`, THEN drop rows with missing minutes, THEN compute points per minute, THEN group by team, THEN average, THEN sort.*

In [ ]:
fake %>%
  filter(!is.na(minutes)) %>%
  mutate(pts_per_min = points / minutes) %>%
  group_by(team) %>%
  summarize(avg_pts_per_min = mean(pts_per_min)) %>%
  arrange(desc(avg_pts_per_min))

### Your turn: chains

These need several functions each. Decide the steps yourself.

**Q13.** Which five teams scored the most points per game on average in the **regular season**? Create `top5`, with columns `team` and `avg_points`, 5 rows, sorted from highest to lowest.

In [ ]:
top5 <- wnba %>%
  filter(game_type == "regular") %>%
  group_by(team) %>%
  summarize(avg_points = mean(points)) %>%
  arrange(desc(avg_points)) %>%
  head(5)
top5

In [ ]:
. = ottr::check("tests/q13.R")

**Q14.** What share of team-games are blowouts (decided by 20+ points) in each **season** and **game type**? Create `blowout.rates`, with columns `season`, `game_type`, and `blowout_share` (8 rows). (Hint: the mean of a `TRUE`/`FALSE` column is the share of `TRUE`s.)

In [ ]:
blowout.rates <- wnba %>%
  mutate(blowout = abs(points - opp_points) >= 20) %>%
  group_by(season, game_type) %>%
  summarize(blowout_share = mean(blowout), .groups = "drop")
blowout.rates

In [ ]:
. = ottr::check("tests/q14.R")

**Q15.** Which teams hosted the most **playoff** games? Create `playoff.hosts`, with columns `team` and `home_games` (the number of playoff games that team played at home), sorted from most to fewest. Only teams that hosted at least one game need to appear.

(Does the answer look familiar? Compare it to what you found in Part C of the section activity.)

In [ ]:
playoff.hosts <- wnba %>%
  filter(game_type == "playoff" & home_away == "home") %>%
  group_by(team) %>%
  summarize(home_games = n()) %>%
  arrange(desc(home_games))
playoff.hosts

In [ ]:
. = ottr::check("tests/q15.R")

### Challenge

**Q16.** In the section activity you used `difference_in_means()` to estimate the home-minus-away gap in points. You can get the same *estimate* with dplyr alone. Create `gap.by.type`, with columns `game_type` and `gap`, where `gap` is the average points of **home** teams minus the average points of **away** teams, separately for regular-season and playoff games. (2 rows.)

*Hint: inside `summarize()`, you can take the mean of a subset of a column using square brackets, like `mean(points[home_away == "home"])`.*

In [ ]:
gap.by.type <- wnba %>%
  group_by(game_type) %>%
  summarize(gap = mean(points[home_away == "home"]) - mean(points[home_away == "away"]))
gap.by.type

In [ ]:
. = ottr::check("tests/q16.R")

---
## What you learned

- `%>%`: pass the result of one step into the next
- `filter()`: keep rows that meet a condition (with `==`, `&`, `|`, `%in%`, `is.na()`)
- `arrange()` / `desc()`: sort rows
- `select()`: keep or drop columns
- `mutate()`: create new columns
- `group_by()` + `summarize()`: one summary row per group, with `n()` to count

Almost every real data analysis is some chain of these steps.